# Purpose/risk over the gold-1000 label set

`PurposeRiskService` loads the `gold1000-r1` label set -- the renderer-1 passages the
gold-1000 annotations were made on -- has the store create a run over it, and streams
every label through the model. One row per label, failed ones included, goes to
`data/dsld/runs/gold1000-r1-<YYMMDDHHMM>-<n>/raw/purpose_risk.jsonl`.

`gold1000-r2` holds the same ids re-rendered with renderer 2, but nothing was annotated
on that text; `gold1000-r1` is the set whose output can be read beside the gold.

The running log is the controller's and the model's `logging` output, sent to this
notebook: one progress line per finished label with the ETA, a line per retry, and a
warning per label given up on.

In [ ]:
import json
import logging
import sys

import pandas as pd

from engine.repositories.dsld import loader, store
from engine.services.purpose_risk.service import PurposeRiskService

LABEL_SET = "gold1000-r1"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    stream=sys.stdout,
    force=True,
)

## The label set and its gold

What the run will read: 1000 flattened labels and the `passages_sha` the run records,
and the gold annotations saved beside them, one per label.

In [ ]:
passages_sha, labels = loader.load_flattened(LABEL_SET)

set_dir = store.LABELS_ROOT / LABEL_SET
with open(set_dir / store.LABEL_SET_ANNOTATIONS_NAME) as f:
    gold = {int(r["dsld_id"]): r for r in map(json.loads, f)}

print(json.loads((set_dir / store.LABEL_SET_MANIFEST_NAME).read_text()))
assert set(gold) == set(labels), "gold and label set cover different ids"
passages_sha, len(labels), len(gold)

## Is the GPU free?

The model needs ~21 GB. Anything listed under compute apps is holding the card.

In [8]:
!nvidia-smi --query-gpu=memory.free --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv

memory.free [MiB]
24111 MiB
pid, process_name, used_gpu_memory [MiB]


## The run

The whole set, in its file order. Leaving the `with` block closes the controller,
which stops the server and frees the card -- also on an error or an interrupt. The
last line printed is the run directory and its counts.

To try it on a few labels first, pass `limit=20`; that is a separate run.

In [ ]:
with PurposeRiskService() as svc:
    svc.run(LABEL_SET)

2026-09-30 21:25:01,545 INFO engine.controllers.qwen_controller 1/1000 1872 tok · 21.3 tok/s · eta 56m18s
2026-09-30 21:25:01,557 INFO engine.controllers.qwen_controller 2/1000 3952 tok · 38.9 tok/s · eta 28m13s
2026-09-30 21:25:03,213 INFO engine.controllers.qwen_controller 3/1000 5965 tok · 28.9 tok/s · eta 27m58s
2026-09-30 21:25:04,222 INFO engine.controllers.qwen_controller 4/1000 8177 tok · 41.1 tok/s · eta 25m08s
2026-09-30 21:25:05,408 INFO engine.controllers.qwen_controller 5/1000 10365 tok · 43.3 tok/s · eta 24m02s
2026-09-30 21:25:07,606 INFO engine.controllers.qwen_controller 6/1000 13127 tok · 52.2 tok/s · eta 26m04s
2026-09-30 21:25:12,101 INFO engine.controllers.qwen_controller 7/1000 15910 tok · 57.0 tok/s · eta 32m57s
2026-09-30 21:25:12,789 INFO engine.controllers.qwen_controller 8/1000 19117 tok · 72.5 tok/s · eta 30m14s
2026-09-30 21:25:14,645 INFO engine.controllers.qwen_controller 9/1000 22048 tok · 79.2 tok/s · eta 30m15s
2026-09-30 21:25:16,261 INFO engine.contr

## The rows

The newest run over the set, one row per label.

In [ ]:
run = max(
    store.RUNS_ROOT.glob(f"{LABEL_SET}-*"),
    key=lambda d: (d.name.rsplit("-", 2)[1], int(d.name.rsplit("-", 1)[1])),
)
print(run)
print(loader.read_manifest(run))

df = pd.DataFrame(loader.read_rows(run))
print(df.status.value_counts().to_string())
df[["dsld_id", "status", "attempts", "failed", "purpose", "risk", "latency_ms"]]

## Run stats

Status and attempts, items per label, the scores the model gave its items, and
tokens and latency per label.

In [6]:
# Status and attempts: how many labels needed a retry, and how many never passed.
print(df.status.value_counts().to_string(), "\n")
print(df.attempts.value_counts().sort_index().rename("labels by attempts").to_string(), "\n")

# How much each label yielded: items per label, and labels with none.
df["n_purpose"] = df.purpose.str.len()
df["n_risk"] = df.risk.str.len()
print(df[["n_purpose", "n_risk"]].describe().round(2).to_string(), "\n")
print(f"no purpose: {(df.n_purpose == 0).sum()}   no risk: {(df.n_risk == 0).sum()}   "
      f"neither: {((df.n_purpose == 0) & (df.n_risk == 0)).sum()}   of {len(df)}\n")

# Scores the model gave its own items.
scores = pd.DataFrame(
    [{"kind": kind, "score": item["score"]} for kind in ("purpose", "risk")
     for items in df[kind] for item in items]
)
print(scores.value_counts().unstack(fill_value=0).to_string(), "\n")

# Tokens and latency per label. Slots run in parallel, so summed latency is not wall time.
print(df[["prompt_tokens", "completion_tokens", "latency_ms"]].describe().round(0).to_string())

status
ok               999
failed_checks      1 

attempts
1    998
2      1
3      1 

       n_purpose   n_risk
count    1000.00  1000.00
mean        2.04     2.14
std         2.28     2.02
min         0.00     0.00
25%         0.00     1.00
50%         1.00     2.00
75%         3.00     3.00
max        17.00    15.00 

no purpose: 251   no risk: 152   neither: 41   of 1000

score       5    4   3  1  2
kind                        
purpose  1858  128  51  2  2
risk     2123   11   4  0  3 

       prompt_tokens  completion_tokens  latency_ms
count         1000.0             1000.0      1000.0
mean          2075.0               79.0      4226.0
std            259.0               67.0      2906.0
min           1653.0                2.0       496.0
25%           1919.0               35.0      2220.0
50%           2018.0               62.0      3512.0
75%           2139.0              101.0      5312.0
max           3861.0              400.0     22632.0


## Model beside gold: counts

Items per label from the model and from the gold, and how often each side has none
where the other has some. Only `ok` rows are counted.

In [ ]:
ok = df[df.status == "ok"].copy()
ok["gold_purpose"] = ok.dsld_id.map(lambda i: len(gold[i]["purpose"]))
ok["gold_risk"] = ok.dsld_id.map(lambda i: len(gold[i]["risk"]))

print(ok[["n_purpose", "gold_purpose", "n_risk", "gold_risk"]].describe().round(2).to_string(), "\n")
for kind in ("purpose", "risk"):
    m, g = ok[f"n_{kind}"] > 0, ok[f"gold_{kind}"] > 0
    print(f"{kind:8s} both: {(m & g).sum():4d}   model only: {(m & ~g).sum():4d}   "
          f"gold only: {(~m & g).sum():4d}   neither: {(~m & ~g).sum():4d}   of {len(ok)}")

## Sample: passage, model, gold

Each sampled label's passage, the items the model took from it with its scores, and
the gold items for the same label.

In [ ]:
# A seeded handful of labels. Change SAMPLE_SEED to see others.
SAMPLE_SEED = 0

for row in df.sample(5, random_state=SAMPLE_SEED).itertuples():
    label = labels[row.dsld_id]
    print(f"=== {row.dsld_id}  {label.name}  [{row.status}, {row.attempts} attempt(s)]")
    print(label.passage[:800] + (" ..." if len(label.passage) > 800 else ""))
    for kind in ("purpose", "risk"):
        print(f"--- {kind}: model")
        for item in getattr(row, kind):
            print(f"  [{item['score']}] {item['text']}")
        print(f"--- {kind}: gold")
        for item in gold[row.dsld_id][kind]:
            print(f"  {item['text']}")
    print()

## Most repeated purpose and risk items

In [ ]:
# The most repeated items across the run, beside the gold's. Repeated risk items are
# mostly generic label warnings; a later filter handles them on the output side.
for kind in ("purpose", "risk"):
    model_counts = pd.Series([item["text"] for items in df[kind] for item in items]).value_counts()
    gold_counts = pd.Series([item["text"] for r in gold.values() for item in r[kind]]).value_counts()
    print(f"--- {kind}: model {model_counts.sum()} items, {len(model_counts)} distinct   "
          f"gold {gold_counts.sum()} items, {len(gold_counts)} distinct")
    print(model_counts.head(10).to_string(), "\n")
    print(gold_counts.head(10).to_string(), "\n")

## One failed label in full

The raw answer is kept on every row, so a failure can be read without the GPU.

In [ ]:
failed = df[df.status != "ok"]
failed.iloc[0].to_dict() if len(failed) else "no failed labels"

## Resume

If the run was interrupted, this reads the run's ids and label set from the run
directory and does only the ids without a row yet. A changed prompt or sampling
setting is refused -- start a new run instead.

In [ ]:
with PurposeRiskService() as svc:
    svc.run(run=run)